# Phase 1 — Core sweep

**Paper 1 · Unrecognized organ damage · AI-READI v3.0.0**

Phase 1 is the paper's backbone: **it publishes even if every Phase-2 track is
null.** Six batches, each independently re-verified by a second implementation
that never imports `aireadi`.

| | Question | Artifact |
|---|---|---|
| **E1.0** | What counts as "abnormal", and what is the denominator? | `E1_0_threshold_spec.csv` |
| **E1.1** | How common is measured damage? | `E1_1_prevalence_by_group.csv` |
| **E1.2** | How much of it went unrecognized? *(headline)* | `E1_2_unrecognized_by_group.csv` |
| **E1.3** | How many organs at once? | `E1_3_organ_counts.csv` |
| **E1.4** | Who is unrecognized, vs who knows? | `E1_4_models.csv` |
| **E1.5** | Does it survive a different cutoff? | `E1_5_threshold_sweep.csv` |

Narrative write-up: `reports/2026-08-12-phase1-report.md`.

> **Reproducing vs re-deriving.** Cells below recompute the headline aggregates
> from the master table through `src/aireadi`, so the work is visible. The
> fitted logistic models (E1.4) are read from the artifact `run_e1_4.py`
> produced rather than refitted here — one fitting implementation, not two.

## Setup

In [ ]:
# Thin-notebook bootstrap: put `src/` and the paper's `scripts/` on the path so
# this notebook can call the same code the command-line runners call. No
# cleaning or threshold logic is defined here -- it all lives in `src/aireadi`.
import sys, pathlib

REPO = pathlib.Path.cwd()
while not (REPO / "src" / "aireadi").exists():          # works from any cwd
    REPO = REPO.parent
sys.path[:0] = [str(REPO / "src"), str(REPO / "papers/p1-unrecognized-damage/scripts")]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from aireadi import figures as fg, results, stats, thresholds
import _phase1

fg.style()
RESULTS = results.results_dir("p1")
GROUPS = fg.SEVERITY_ORDER
pd.set_option("display.width", 180)
print("repo:", REPO.name)


In [ ]:
df = _phase1.load()
print(f"{len(df):,} participants")

## E1.0 — Fixing the definitions before counting anything

The single easiest place for a paper to lose a reviewer is deciding what counts
as "abnormal" after seeing what the counts do. So it was settled and written
down first, and every cutoff is swept in E1.5.

| Organ | Marker | Abnormal | Why this line |
|---|---|---|---|
| Kidney | urine ACR | **≥ 30 mg/g** | KDIGO category A2, the standard screening threshold. Needs no sex variable. |
| Heart | hs-cTnT | **≥ 14 ng/L** | Sex-neutral 99th-percentile upper reference limit. Sex-specific limits (~10 F / ~15–16 M) are impossible — the public release removes sex. |
| Nerve | monofilament, worse foot | **≥ 2 insensate of 10** | Guideline "loss of protective sensation" was written for 3–4 site exams; on a 10-site exam one equivocal miss would qualify. |

**Unrecognized fraction** = (abnormal **and** self-report says no) ÷ (abnormal
**and** the item was answered). A refusal is never recoded as "no" — "never
told" and "would not say" are different things, and that difference is the
paper. The refusals-included denominator is reported beside it every time.

In [ ]:
spec = []
for organ, col in [("kidney", "acr_mg_g"), ("heart", "troponin_t"),
                   ("nerve", "monofilament_min")]:
    abn = df[f"abn_{organ}"]
    row = {"organ": organ, "n_measured": int(abn.notna().sum()),
           "n_abnormal": int(abn.eq(1).sum())}
    sr = df.get(f"sr_{organ}")
    if sr is not None:
        row["n_refused_among_abnormal"] = int((abn.eq(1) & sr.isna()).sum())
        row["unrec_denominator"] = int((abn.eq(1) & sr.notna()).sum())
    else:
        row["n_refused_among_abnormal"] = None
        row["unrec_denominator"] = None      # no comparator -- E0.GATE
    spec.append(row)

pd.DataFrame(spec).to_string(index=False)

## E1.1 — How much measured damage is there?

Counting, not testing: these numbers publish whatever they say. The trend test
asks the *ordered* question the paper actually claims ("does it rise with
severity"), rather than the weaker "differs across groups".

In [ ]:
df["abn_any"] = df.n_organs_abnormal.gt(0).astype(float).mask(df.n_organs_abnormal.isna())

prev = pd.concat([
    stats.proportion_by_group(df, f"abn_{o}").assign(organ=o)
    for o in [*thresholds.ORGANS, "any"]
])
prev.loc["Overall"][["organ", "n", "k", "pct", "ci_lo", "ci_hi", "trend_z", "trend_p"]]

### Figure — damage rises across the whole severity spectrum

Severity group is **ordered**, so it gets a single-hue ramp light→dark rather
than four unrelated colours: the ordering is the finding.

Look at the leftmost bar of each panel. **A quarter of the "healthy" control
arm already has an abnormal result** — arguably the strongest screening
argument in the paper, and a warning that "healthy" is not a clean baseline for
any Phase-2 comparison.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 4), sharey=True)
for ax, organ in zip(axes, ["kidney", "heart", "nerve", "any"]):
    sub = prev[prev.organ == organ]
    vals = [sub.loc[g, "pct"] for g in GROUPS]
    yerr = [[sub.loc[g, "pct"] - sub.loc[g, "ci_lo"] for g in GROUPS],
            [sub.loc[g, "ci_hi"] - sub.loc[g, "pct"] for g in GROUPS]]
    rects = ax.bar(range(4), vals, 0.66, color=fg.SEVERITY,
                   edgecolor=fg.SURFACE, linewidth=1.2)
    ax.errorbar(range(4), vals, yerr=yerr, fmt="none", ecolor=fg.INK_SECONDARY,
                elinewidth=1.2, capsize=3)
    for r, v in zip(rects, vals):
        ax.annotate(f"{v:.1f}%", xy=(r.get_x() + r.get_width() / 2, v),
                    xytext=(0, 9), textcoords="offset points", ha="center",
                    fontsize=8.5, color=fg.INK_SECONDARY)
    o = sub.loc["Overall"]
    ax.set_title(f"{fg.ORGAN_LABEL[organ]}   {o.pct}% overall", loc="left")
    ax.set_xticks(range(4))
    ax.set_xticklabels(GROUPS, rotation=45, ha="right", fontsize=8.5)
    ax.annotate(f"trend p = {o.trend_p:.0e}", xy=(0.03, 0.93),
                xycoords="axes fraction", fontsize=8, color=fg.MUTED)
axes[0].set_ylabel("% with an abnormal result"); axes[0].set_ylim(0, 78)

fg.finish(fig, "E1.1 \u2014 Measured organ damage rises across the severity spectrum",
          "Every organ trends upward; bars are Wilson 95% CIs. Note the leftmost bar of "
          "each panel \u2014 the 'healthy' control group is already a quarter abnormal.",
          "Source: results/E1_1_prevalence_by_group.csv")
fig.savefig(RESULTS / "E1_1_prevalence_figure.png", dpi=200, bbox_inches="tight")
fig

## E1.2 — How much went unrecognized? *(the headline)*

For kidney and heart we can ask the paper's real question. *(Nerve cannot
answer it — `E0.GATE`.)*

Two quantities, and they are **not** the same question:

- **Conditional fraction** — of people who *have* damage, what share were never told?
- **Population burden** — of *everyone* in the group, what share carries unrecognized damage?

They move in **opposite directions** across severity. Both are reported, and
they multiply out exactly: prevalence × fraction = burden.

In [ ]:
unrec = pd.concat([stats.proportion_by_group(df, f"unrec_{o}").assign(organ=o)
                   for o in thresholds.UNRECOGNIZED_ORGANS])

# "Either organ": the definition lives in the package, NOT here. Writing it out
# by hand is how this notebook first produced 625/478 against the runner's
# 615/471 -- the rule is "both organs evaluable", not "either".
e = thresholds.either_organ(df)
df["unrec_either"] = e.unrecognized.where(e.answered & e.abnormal)
unrec = pd.concat([unrec, stats.proportion_by_group(df, "unrec_either").assign(organ="either")])

# Burden: the same numerator over EVERYONE evaluable, not just the abnormal.
for organ in ["kidney", "heart"]:
    df[f"burden_{organ}"] = np.where(
        df[f"abn_{organ}"].notna() & df[f"sr_{organ}"].notna(),
        (df[f"abn_{organ}"].eq(1) & df[f"sr_{organ}"].eq(0)).astype(float), np.nan)
df["burden_either"] = e.unrecognized.where(e.answered)   # same numerator, everyone evaluable
burden = pd.concat([stats.proportion_by_group(df, f"burden_{o}").assign(organ=o)
                    for o in ["kidney", "heart", "either"]])

print("UNRECOGNIZED FRACTION (of those abnormal)")
print(unrec.loc["Overall"][["organ", "n", "k", "pct", "ci_lo", "ci_hi", "trend_z"]].to_string(index=False))
print("\nPOPULATION BURDEN (of everyone evaluable)")
print(burden.loc["Overall"][["organ", "n", "k", "pct", "trend_z"]].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
SERIES = {"kidney": fg.ORGAN["kidney"], "heart": fg.ORGAN["heart"],
          "either": fg.ORGAN["nerve"]}
for ax, tab, title, ylab in [
        (axes[0], unrec, "The FRACTION unrecognized falls",
         "% never told\n\u2014 of those who ARE abnormal"),
        (axes[1], burden, "\u2026while the BURDEN rises",
         "% carrying unrecognized damage\n\u2014 of EVERYONE in the group")]:
    for organ, color in SERIES.items():
        vals = [tab[tab.organ == organ].loc[g, "pct"] for g in GROUPS]
        ax.plot(range(4), vals, marker="o", color=color, label=fg.ORGAN_LABEL[organ])
        ax.annotate(f"{vals[-1]:.1f}%", xy=(3, vals[-1]), xytext=(8, 0),
                    textcoords="offset points", fontsize=9, color=color,
                    fontweight="bold", va="center")
    ax.set_xticks(range(4)); ax.set_xticklabels(GROUPS, fontsize=9)
    ax.set_xlim(-0.25, 3.6); ax.set_ylabel(ylab, fontsize=8.5)
    ax.set_title(title, loc="left")
axes[0].set_ylim(45, 95); axes[1].set_ylim(0, 48)
axes[0].legend(loc="lower left", ncol=3)

fg.finish(fig, "E1.2 \u2014 Two true statements that point in opposite directions",
          "Same data, two questions. Reporting only one would be misleading, so both are "
          "computed; prevalence \u00d7 fraction = burden, exactly.",
          "Source: results/E1_2_unrecognized_by_group.csv, E1_2_population_burden.csv")
fig.savefig(RESULTS / "E1_2_unrecognized_figure.png", dpi=200, bbox_inches="tight")
fig

### The 2×2, not just the striking cell

Reviewers will ask for the whole table. The third row is the honest one: most
people who *report* a kidney or heart diagnosis tested **normal** that day.
That is not a contradiction — a treated or past condition should test normal —
but it is exactly why the paper must say **"unrecognized abnormal findings that
warrant follow-up"** and never "undiagnosed disease".

In [ ]:
rows = []
for organ in ["kidney", "heart"]:
    abn, sr = df[f"abn_{organ}"], df[f"sr_{organ}"]
    ok = abn.notna() & sr.notna()
    rows.append({
        "organ": organ,
        "abnormal, not reported": int((ok & abn.eq(1) & sr.eq(0)).sum()),
        "abnormal, reported": int((ok & abn.eq(1) & sr.eq(1)).sum()),
        "normal, reported": int((ok & abn.eq(0) & sr.eq(1)).sum()),
        "normal, not reported": int((ok & abn.eq(0) & sr.eq(0)).sum()),
        "total evaluable": int(ok.sum()),
    })
conc = pd.DataFrame(rows).set_index("organ")
print(conc.to_string())
for organ in ["kidney", "heart"]:
    r = conc.loc[organ]
    reported = r["abnormal, reported"] + r["normal, reported"]
    print(f"\n{organ}: of {reported} reporting a diagnosis, "
          f"{r['normal, reported']} ({r['normal, reported'] / reported:.0%}) tested normal that day")

## E1.3 — How many organs at once?

Restricted to participants measured on all three, so a partial row cannot
masquerade as a clean negative.

In [ ]:
n_org = df.n_organs_abnormal.dropna()
counts = (df.dropna(subset=["n_organs_abnormal"])
            .groupby("study_group_label").n_organs_abnormal
            .value_counts(normalize=True).mul(100).unstack(fill_value=0)
            .reindex(GROUPS).round(1))
counts["mean_organs"] = (df.dropna(subset=["n_organs_abnormal"])
                           .groupby("study_group_label").n_organs_abnormal.mean()
                           .reindex(GROUPS).round(3))
print(f"Measured on all three: {len(n_org):,}")
print(f"Overall: {(n_org == 0).mean():.1%} none, {(n_org >= 2).mean():.1%} two or more\n")
counts

In [ ]:
fig, ax = fg.new_figure(8.4, 4.6)
fg.stacked_bars(ax, GROUPS, {"No organ": counts[0.0], "One": counts[1.0],
                             "Two": counts[2.0], "All three": counts[3.0]},
                fg.SEVERITY)
ax.set_ylabel("% of participants measured on all three"); ax.set_ylim(0, 113)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.09), ncol=4)
for i, g in enumerate(GROUPS):
    ax.annotate(f"mean {counts.loc[g, 'mean_organs']:.2f} organs", xy=(i, 102),
                ha="center", fontsize=8.5, color=fg.INK_SECONDARY)

fg.finish(fig, "E1.3 \u2014 Multi-organ damage concentrates at the severe end",
          "The mean number of damaged organs more than triples, 0.33 \u2192 1.05. Only a third "
          "of insulin-treated participants are clear on all three.",
          "Source: results/E1_3_organ_counts.csv")
fig.savefig(RESULTS / "E1_3_organ_counts_figure.png", dpi=200, bbox_inches="tight")
fig

## E1.4 — Who is unrecognized? *(the section that needed correcting)*

Three nested logistic models, fitted in `run_e1_4.py` and read here:

- **A** — age + severity + site
- **B** — A + HbA1c + BMI
- **C** — B + log marker magnitude

**The two organs do not behave the same way, and an earlier draft of the report
missed it.** A forest plot makes the divergence unmissable in a way the
coefficient table did not.

In [ ]:
models = pd.read_csv(RESULTS / "E1_4_models.csv")
MODELS = ["A: age + severity + site", "B: A + HbA1c + BMI", "C: B + marker magnitude"]
INSULIN = "C(study_group_label)[T.Insulin]"

rows = [models[(models.organ == o) & (models.model == m) & (models.term == INSULIN)].iloc[0]
        .to_dict() | {"organ": o, "short": s}
        for o in ["kidney", "heart"]
        for m, s in zip(MODELS, ["A  age+severity+site", "B  + HbA1c + BMI",
                                 "C  + marker magnitude"])]
sev = pd.DataFrame(rows)
print(sev[["organ", "short", "odds_ratio", "ci_lo", "ci_hi", "p"]].to_string(index=False))

In [ ]:
fig, ax = fg.new_figure(9.6, 4.4)
fg.forest(ax, [f"{r.organ.capitalize()}  \u00b7  {r.short}" for r in sev.itertuples()],
          sev.odds_ratio, sev.ci_lo, sev.ci_hi,
          colors=[fg.ORGAN[o] for o in sev.organ])
ax.set_xlabel("odds of being UNRECOGNIZED \u2014 Insulin vs Healthy (log scale)")
ax.axhline(2.5, color=fg.GRID, linewidth=1.0)
ax.annotate("solid = significant \u00b7 hollow = interval crosses 1", xy=(0.99, 0.96),
            xycoords="axes fraction", ha="right", va="top", fontsize=8, color=fg.MUTED)

fg.finish(fig, "E1.4 \u2014 The severity effect survives adjustment for kidney, but not for heart",
          "Each row adds covariates to the row above. Kidney tightens and stays significant; "
          "heart loses significance once HbA1c, BMI and marker magnitude enter.",
          "Source: results/E1_4_models.csv")
fig.savefig(RESULTS / "E1_4_forest_figure.png", dpi=200, bbox_inches="tight")
fig

**What the figure shows, and why it matters.**

- **Kidney** (blue): the interval sits far left of 1 and *tightens* as covariates
  are added. The severity effect is not explained away by how bad the damage is.
- **Heart** (orange): significant in model A, then the marker goes **hollow** —
  the interval crosses 1 once HbA1c/BMI and marker magnitude enter. For the
  heart, the predictor that holds in all three models is simply **age**
  (OR 0.96/yr, p ≈ 0.00017).

So the §4 finding — *the unrecognized fraction falls as severity rises* — is
**robust for kidney and suggestive for heart**. Writing both as equally
established is the single easiest thing for a reviewer to take apart.

**Carried into Phase 2:** any track testing something against *unrecognized
status* must adjust for **age** as well as marker magnitude, or it will
re-find this and read it as new.

## E1.5 — Does any of this survive a different cutoff?

Every headline was re-run at every rung of every grid — 16 re-analyses.
**Six of seven conclusions hold everywhere. One flips.**

In [ ]:
sweep = pd.read_csv(RESULTS / "E1_5_threshold_sweep.csv")
sweep[sweep.unrecognized_pct.notna()][
    ["organ", "cutoff", "is_primary", "n_abnormal", "prevalence_pct", "unrecognized_pct"]
].to_string(index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8))
for ax, organ, xlabel, primary in [(axes[0], "kidney", "ACR cutoff (mg/g)", "30.0"),
                                   (axes[1], "heart", "hs-cTnT cutoff (ng/L)", "14.0")]:
    sub = sweep[(sweep.organ == organ) & sweep.unrecognized_pct.notna()]
    x = range(len(sub))
    ax.plot(x, sub.unrecognized_pct, marker="o", color=fg.ORGAN[organ])
    ax.axhline(50, color=fg.BASELINE, linewidth=1.2, linestyle="--")
    # Left-aligned: both series START high and descend, so the left end of the
    # 50% line is the clear side. (Right-aligned collides with the kidney curve
    # where it crosses 50 -- the whole point of the panel.)
    ax.annotate("50% \u2014 'most people don't know'", xy=(0.02, 50),
                xycoords=("axes fraction", "data"), xytext=(0, 5),
                textcoords="offset points", ha="left", fontsize=8, color=fg.MUTED)
    for xi, (_, r) in zip(x, sub.iterrows()):
        chosen = str(r.cutoff) == primary
        ax.annotate(f"{r.unrecognized_pct:.1f}%", xy=(xi, r.unrecognized_pct),
                    xytext=(0, 9), textcoords="offset points", ha="center",
                    fontsize=8.5, fontweight="bold" if chosen else "normal",
                    color=fg.INK if chosen else fg.INK_SECONDARY)
        if chosen:
            ax.plot([xi], [r.unrecognized_pct], marker="o", markersize=14,
                    markerfacecolor="none", markeredgecolor=fg.INK, markeredgewidth=1.8)
    ax.set_xticks(list(x)); ax.set_xticklabels(sub.cutoff, fontsize=8.5)
    ax.set_xlabel(xlabel); ax.set_ylim(20, 92)
    ax.set_title(fg.ORGAN_LABEL[organ], loc="left")
axes[0].set_ylabel("% of abnormal results\nthat were unrecognized")
# Both curves fall left-to-right, so the lower-left quadrant is empty in each
# panel -- park the verdict there rather than beside the 50% line.
axes[0].annotate("FLIPS below 50%\nat severe albuminuria", xy=(0.03, 0.06),
                 xycoords="axes fraction", ha="left", va="bottom",
                 fontsize=8.5, color="#d03b3b", fontweight="bold")
axes[1].annotate("never crosses 50%\n\u2014 holds at every cutoff", xy=(0.03, 0.06),
                 xycoords="axes fraction", ha="left", va="bottom",
                 fontsize=8.5, color="#0ca30c", fontweight="bold")

fg.finish(fig, "E1.5 \u2014 One claim depends on where the line is drawn",
          "Circled marker = the cutoff we chose \u2014 mid-range, not the flattering end. "
          "Kidney's 'majority unrecognized' flips at severe albuminuria; heart holds throughout.",
          "Source: results/E1_5_threshold_sweep.csv")
fig.savefig(RESULTS / "E1_5_sweep_figure.png", dpi=200, bbox_inches="tight")
fig

**The flip bounds the claim rather than breaking it.** At ACR ≥ 300 (severe
albuminuria) *two-thirds of people do know* — which is clinically sensible,
since heavy protein leakage produces symptoms and referrals. So the paper's
claim becomes:

> The unrecognized finding is about **mild-to-moderate, early damage** —
> exactly the damage screening is supposed to catch and treatment can still
> reverse. It is not a claim that severe kidney disease goes undiagnosed.

That is a *better* screening argument than the vaguer version. It belongs in
the supplement as a figure, not a footnote.

Heart holds across its entire range (59.2–79.0%), which matters because
troponin is the cutoff we are least sure of — the public release removes sex,
so the correct sex-specific limits cannot be applied.

## Where Phase 1 leaves the paper

**Headline:** of 615 participants with kidney or heart damage on a same-day
test, **471 (76.6%) reported no corresponding diagnosis.**

Four things Phase 2 must carry:

1. **"Damage" and "unrecognized damage" behave differently** — opposite
   directions across severity. Any track testing against unrecognized status
   must say which it means and adjust for severity.
2. **Marker magnitude is required as a covariate — and for heart, so is age.**
3. **The healthy group is not a clean control.** A quarter have an abnormal
   result, which shrinks any effect measured against it.
4. **Small cells are coming.** Insulin × abnormal × unrecognized is already
   43–77 people. Bootstrap anything claimed from a cell under ~50.

Every run above, including the nulls, is in `RESULTS_LOG.md`. Verification:
`scripts/verify/verify_e1_*.py` recompute each number from the raw CSVs without
importing `aireadi`; `verify_report.py` traces every number quoted in the
narrative report back to a committed artifact.